In [ ]:

!pip install -q datasets pyarrow


In [ ]:

import pandas as pd
import numpy as np
from datasets import load_dataset

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

url = "https://dataset.raid-bench.xyz/train.csv"

human_chunks = []

# Read RAID in chunks so the full dataset does not need to sit in memory
for chunk in pd.read_csv(url, chunksize=100000):

    # Keep only HUMAN-written rows
    human_chunk = chunk[
        chunk["model"].astype(str).str.strip().str.lower().eq("human")
    ].copy()

    human_chunks.append(human_chunk)

# Combine all human rows
human_df = pd.concat(human_chunks, ignore_index=True)

print("Human data shape:", human_df.shape)

print("\nModels:")
print(human_df["model"].value_counts())

print("\nDomains:")
print(human_df["domain"].value_counts())

display(human_df.head())

Human data shape: (160452, 11)

Models:
model
human    160452
Name: count, dtype: int64

Domains:
domain
books        21372
news         21360
wiki         21348
reddit       21348
recipes      21264
poetry       21252
abstracts    21192
reviews      11316
Name: count, dtype: int64


,id,adv_source_id,source_id,model,decoding,repetition_penalty,attack,domain,title,prompt,generation
0,e5e058ce-be2b-459d-af36-32532aaba5ff,e5e058ce-be2b-459d-af36-32532aaba5ff,e5e058ce-be2b-459d-af36-32532aaba5ff,human,NaN,NaN,none,abstracts,FUTURE-AI: Guiding Principles and Consensus Recommendations for Trustworthy Artificial Intelligence in Future Medica...,NaN,The recent advancements in artificial intelligence (AI) combined with the\nextensive amount of data generated by tod...
1,f95b107b-d176-4af5-90f7-4d0bb20caf93,f95b107b-d176-4af5-90f7-4d0bb20caf93,f95b107b-d176-4af5-90f7-4d0bb20caf93,human,NaN,NaN,none,abstracts,EdgeFlow: Achieving Practical Interactive Segmentation with Edge-Guided Flow,NaN,"High-quality training data play a key role in image segmentation tasks.\nUsually, pixel-level annotations are expens..."
2,856d8972-9e3d-4544-babc-0fe16f21e04d,856d8972-9e3d-4544-babc-0fe16f21e04d,856d8972-9e3d-4544-babc-0fe16f21e04d,human,NaN,NaN,none,abstracts,Semi-supervised Contrastive Learning for Label-efficient Medical Image Segmentation,NaN,The success of deep learning methods in medical image segmentation tasks\nheavily depends on a large amount of label...
3,fbc8a5ea-90fa-47b8-8fa7-73dd954f1524,fbc8a5ea-90fa-47b8-8fa7-73dd954f1524,fbc8a5ea-90fa-47b8-8fa7-73dd954f1524,human,NaN,NaN,none,abstracts,Combo Loss: Handling Input and Output Imbalance in Multi-Organ Segmentation,NaN,Simultaneous segmentation of multiple organs from different medical imaging\nmodalities is a crucial task as it can ...
4,72c41b8d-0069-4886-b734-a4000ffca286,72c41b8d-0069-4886-b734-a4000ffca286,72c41b8d-0069-4886-b734-a4000ffca286,human,NaN,NaN,none,abstracts,Attention-Based 3D Seismic Fault Segmentation Training by a Few 2D Slice Labels,NaN,"Detection faults in seismic data is a crucial step for seismic structural\ninterpretation, reservoir characterizatio..."


In [ ]:
print(human_df.columns.tolist())

['id', 'adv_source_id', 'source_id', 'model', 'decoding', 'repetition_penalty', 'attack', 'domain', 'title', 'prompt', 'generation']


In [ ]:
keep_cols = [
    "id",
    "source_id",
    "domain",
    "title",
    "generation",
    "model",
    "attack"
]

keep_cols = [
    col for col in keep_cols
    if col in human_df.columns
]

human_df = human_df[keep_cols].copy()

# Rename text column
human_df = human_df.rename(
    columns={"generation": "text"}
)

# Remove empty text
human_df["text"] = (
    human_df["text"]
    .astype("string")
    .str.strip()
)

human_df = human_df[
    human_df["text"].notna() &
    human_df["text"].ne("")
].copy()

print("Clean human data:", human_df.shape)

display(human_df.head())

Clean human data: (160452, 7)


,id,source_id,domain,title,text,model,attack
0,e5e058ce-be2b-459d-af36-32532aaba5ff,e5e058ce-be2b-459d-af36-32532aaba5ff,abstracts,FUTURE-AI: Guiding Principles and Consensus Recommendations for Trustworthy Artificial Intelligence in Future Medica...,The recent advancements in artificial intelligence (AI) combined with the extensive amount of data generated by toda...,human,none
1,f95b107b-d176-4af5-90f7-4d0bb20caf93,f95b107b-d176-4af5-90f7-4d0bb20caf93,abstracts,EdgeFlow: Achieving Practical Interactive Segmentation with Edge-Guided Flow,"High-quality training data play a key role in image segmentation tasks. Usually, pixel-level annotations are expensi...",human,none
2,856d8972-9e3d-4544-babc-0fe16f21e04d,856d8972-9e3d-4544-babc-0fe16f21e04d,abstracts,Semi-supervised Contrastive Learning for Label-efficient Medical Image Segmentation,The success of deep learning methods in medical image segmentation tasks heavily depends on a large amount of labele...,human,none
3,fbc8a5ea-90fa-47b8-8fa7-73dd954f1524,fbc8a5ea-90fa-47b8-8fa7-73dd954f1524,abstracts,Combo Loss: Handling Input and Output Imbalance in Multi-Organ Segmentation,Simultaneous segmentation of multiple organs from different medical imaging modalities is a crucial task as it can b...,human,none
4,72c41b8d-0069-4886-b734-a4000ffca286,72c41b8d-0069-4886-b734-a4000ffca286,abstracts,Attention-Based 3D Seismic Fault Segmentation Training by a Few 2D Slice Labels,"Detection faults in seismic data is a crucial step for seismic structural interpretation, reservoir characterization...",human,none


In [ ]:
AI_DATA_PATH = (
    "/content/drive/MyDrive/Linguistics_feature/raid_chatgpt_only.csv"
)

chatgpt_df = pd.read_csv(AI_DATA_PATH)

In [ ]:
experiment_source_ids = (
    chatgpt_df["source_id"]
    .dropna()
    .astype(str)
    .unique()
)

print(
    "Unique source IDs in ChatGPT experiment:",
    len(experiment_source_ids)
)

Unique source IDs in ChatGPT experiment: 13371


In [ ]:
human_matched = human_df[
    human_df["source_id"]
    .astype(str)
    .isin(experiment_source_ids)
].copy()

human_matched = (
    human_matched
    .drop_duplicates(subset=["source_id"])
    .reset_index(drop=True)
)

print("Matched human samples:", len(human_matched))

print("\nMatched human samples by domain:")
print(
    human_matched["domain"]
    .value_counts()
    .sort_index()
)

display(human_matched.head())

Matched human samples: 13371

Matched human samples by domain:
domain
abstracts    1766
books        1781
news         1780
poetry       1771
recipes      1772
reddit       1779
reviews       943
wiki         1779
Name: count, dtype: int64


,id,source_id,domain,title,text,model,attack
0,e5e058ce-be2b-459d-af36-32532aaba5ff,e5e058ce-be2b-459d-af36-32532aaba5ff,abstracts,FUTURE-AI: Guiding Principles and Consensus Recommendations for Trustworthy Artificial Intelligence in Future Medica...,The recent advancements in artificial intelligence (AI) combined with the extensive amount of data generated by toda...,human,none
1,f95b107b-d176-4af5-90f7-4d0bb20caf93,f95b107b-d176-4af5-90f7-4d0bb20caf93,abstracts,EdgeFlow: Achieving Practical Interactive Segmentation with Edge-Guided Flow,"High-quality training data play a key role in image segmentation tasks. Usually, pixel-level annotations are expensi...",human,none
2,856d8972-9e3d-4544-babc-0fe16f21e04d,856d8972-9e3d-4544-babc-0fe16f21e04d,abstracts,Semi-supervised Contrastive Learning for Label-efficient Medical Image Segmentation,The success of deep learning methods in medical image segmentation tasks heavily depends on a large amount of labele...,human,none
3,fbc8a5ea-90fa-47b8-8fa7-73dd954f1524,fbc8a5ea-90fa-47b8-8fa7-73dd954f1524,abstracts,Combo Loss: Handling Input and Output Imbalance in Multi-Organ Segmentation,Simultaneous segmentation of multiple organs from different medical imaging modalities is a crucial task as it can b...,human,none
4,72c41b8d-0069-4886-b734-a4000ffca286,72c41b8d-0069-4886-b734-a4000ffca286,abstracts,Attention-Based 3D Seismic Fault Segmentation Training by a Few 2D Slice Labels,"Detection faults in seismic data is a crucial step for seismic structural interpretation, reservoir characterization...",human,none


In [ ]:
HUMAN_MATCHED_PATH = (
    "/content/drive/MyDrive/Linguistics_feature/"
    "raid_human_matched_to_chatgpt.csv"
)

human_matched.to_csv(
    HUMAN_MATCHED_PATH,
    index=False
)

print("Saved to:")
print(HUMAN_MATCHED_PATH)

Saved to:
/content/drive/MyDrive/Linguistics_feature/raid_human_matched_to_chatgpt.csv
